# Sentiment-Price Correlation Tool

This notebook covers the data acquisition and preprocessing stages only.

It collects historical NVIDIA stock data and Yahoo Finance news, then cleans the news dataset so it is ready for later sentiment analysis work.

In [1]:
import pandas as pd
import numpy as np
import requests
import yfinance as yf
from bs4 import BeautifulSoup

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Project Configuration

Define the notebook parameters once so the rest of the pipeline can reuse them consistently.

In [2]:
ticker = "NVDA"
start_date = "2020-01-01"
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")
raw_data_dir = "data/raw"
processed_data_dir = "data/processed"

project_config = {
    "ticker": ticker,
    "start_date": start_date,
    "end_date": end_date,
    "raw_data_dir": raw_data_dir,
    "processed_data_dir": processed_data_dir,
}

project_config

{'ticker': 'NVDA',
 'start_date': '2020-01-01',
 'end_date': '2026-07-15',
 'raw_data_dir': 'data/raw',
 'processed_data_dir': 'data/processed'}

## 3. Download Historical Stock Data

Pull NVIDIA price history with `yfinance` and keep the core OHLCV fields for later analysis.

In [3]:
# Download the historical daily price data and keep the key market columns.
stock_df = yf.download(
    tickers=ticker,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False,
)

if stock_df.empty:
    raise ValueError(f"No stock data returned for {ticker}.")

stock_df = stock_df.reset_index()
stock_df = stock_df.loc[:, ["Date", "Open", "High", "Low", "Close", "Volume"]]

print("Head:")
print(stock_df.head().to_string(index=False))
print("\nInfo:")
stock_df.info()
print("\nShape:", stock_df.shape)
print("\nDescribe:")
print(stock_df.describe().to_string())

Head:
      Date    Open    High     Low   Close    Volume
              NVDA    NVDA    NVDA    NVDA      NVDA
2020-01-02 5.96875 5.99775 5.91800 5.99775 237536000
2020-01-03 5.87750 5.94575 5.85250 5.90175 205384000
2020-01-06 5.80800 5.93175 5.78175 5.92650 262636000
2020-01-07 5.95500 6.04425 5.90975 5.99825 314856000
2020-01-08 5.99400 6.05100 5.95375 6.00950 277108000

Info:
<class 'pandas.DataFrame'>
RangeIndex: 1640 entries, 0 to 1639
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype        
---  ------          --------------  -----        
 0   (Date, )        1640 non-null   datetime64[s]
 1   (Open, NVDA)    1640 non-null   float64      
 2   (High, NVDA)    1640 non-null   float64      
 3   (Low, NVDA)     1640 non-null   float64      
 4   (Close, NVDA)   1639 non-null   float64      
 5   (Volume, NVDA)  1640 non-null   int64        
dtypes: datetime64[s](1), float64(4), int64(1)
memory usage: 77.0 KB

Shape: (1640, 6)

Describe:
Price          

## 4. Download Yahoo Finance News

Use a reusable helper to collect Yahoo Finance news articles for NVDA and return them as a DataFrame.

In [4]:
def fetch_yahoo_finance_news(ticker: str, news_count: int = 20) -> pd.DataFrame:
    """Fetch Yahoo Finance news for a ticker and return a normalized DataFrame."""
    news_items = yf.Ticker(ticker).news or []

    records = []
    for item in news_items[:news_count]:
        content = item.get("content", {})
        records.append(
            {
                "headline": BeautifulSoup(content.get("title", ""), "lxml").get_text(" ", strip=True),
                "publisher": BeautifulSoup(content.get("provider", {}).get("displayName", ""), "lxml").get_text(
                    " ", strip=True
                ),
                "published_date": content.get("pubDate"),
                "article_url": content.get("clickThroughUrl", {}).get("url")
                or content.get("canonicalUrl", {}).get("url"),
            }
        )

    news_df = pd.DataFrame(records, columns=["headline", "publisher", "published_date", "article_url"])
    return news_df


news_df = fetch_yahoo_finance_news(ticker)

if news_df.empty:
    raise ValueError(f"No news articles returned for {ticker}.")

print("Head:")
print(news_df.head().to_string(index=False))
print("\nShape:", news_df.shape)
print("\nInfo:")
news_df.info()

Head:
                                                                                   headline           publisher       published_date                                                                                                                      article_url
         New York enacts statewide moratorium on AI data center construction: A closer look Yahoo Finance Video 2026-07-14T21:09:12Z https://finance.yahoo.com/video/new-york-enacts-statewide-moratorium-on-ai-data-center-construction-a-closer-look-210912459.html
                            Chinese AI model developer DeepSeek preps for IPO: What to know Yahoo Finance Video 2026-07-14T20:57:29Z                    https://finance.yahoo.com/video/chinese-ai-model-developer-deepseek-preps-for-ipo-what-to-know-205729630.html
          This Is the Most Expensive Stock Market in 26 Years. Should Investors Be Worried?         Motley Fool 2026-07-15T10:10:00Z                                  https://finance.yahoo.com/markets/stocks/a

## 5. Data Cleaning

Standardize the news dataset so it is ready for downstream sentiment analysis work.

In [5]:
# Clean the raw news data with vectorized pandas operations only.
news_cleaned_df = news_df.copy()
news_cleaned_df = news_cleaned_df.dropna(subset=["headline", "publisher", "published_date", "article_url"])
news_cleaned_df = news_cleaned_df.assign(
    headline=lambda frame: frame["headline"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip(),
    publisher=lambda frame: frame["publisher"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip(),
    article_url=lambda frame: frame["article_url"].astype("string").str.strip(),
)
news_cleaned_df["published_date"] = pd.to_datetime(news_cleaned_df["published_date"], errors="coerce", utc=True).dt.tz_convert(None)
news_cleaned_df = news_cleaned_df.dropna(subset=["published_date"])
news_cleaned_df = news_cleaned_df.drop_duplicates(subset=["headline"])
news_cleaned_df = news_cleaned_df.sort_values("published_date").reset_index(drop=True)
news_cleaned_df = news_cleaned_df.astype(
    {
        "headline": "string",
        "publisher": "string",
        "article_url": "string",
    }
)

news_cleaned_df

,headline,publisher,published_date,article_url
0,Chinese AI model developer DeepSeek preps for ...,Yahoo Finance Video,2026-07-14 20:57:29,https://finance.yahoo.com/video/chinese-ai-mod...
1,New York enacts statewide moratorium on AI dat...,Yahoo Finance Video,2026-07-14 21:09:12,https://finance.yahoo.com/video/new-york-enact...
2,Can Shiba Inu Reach $1 by 2027? The Answer Wil...,Motley Fool,2026-07-15 09:41:00,https://finance.yahoo.com/markets/crypto/artic...
3,The Stock Market Just Flashed a Clear Warning ...,Motley Fool,2026-07-15 09:50:00,https://finance.yahoo.com/markets/stocks/artic...
4,The Vanguard S&P 500 ETF Is a Great Investment...,Motley Fool,2026-07-15 09:56:00,https://finance.yahoo.com/markets/stocks/artic...
5,"Bitcoin ETFs Are Seeing Massive Outflows, but ...",Motley Fool,2026-07-15 10:00:00,https://finance.yahoo.com/markets/crypto/artic...
6,How Meta Platforms (META) Is Strengthening Its...,Insider Monkey,2026-07-15 10:00:11,https://finance.yahoo.com/technology/ai/articl...
7,Canadian startup that raised US$75 billion wan...,Financial Post,2026-07-15 10:00:56,https://finance.yahoo.com/news/canadian-startu...
8,STMicroelectronics (ENXTPA:STMPA) Could Be 7% ...,Simply Wall St.,2026-07-15 10:09:08,https://finance.yahoo.com/markets/stocks/artic...
9,This Is the Most Expensive Stock Market in 26 ...,Motley Fool,2026-07-15 10:10:00,https://finance.yahoo.com/markets/stocks/artic...


In [6]:
from pathlib import Path

news_output_dir = Path("data")
news_output_dir.mkdir(parents=True, exist_ok=True)
news_output_path = news_output_dir / "nvidia_news.csv"

news_cleaned_df.to_csv(news_output_path, index=False)
print(f"Saved cleaned news to {news_output_path}")

Saved cleaned news to data/nvidia_news.csv


## 5.1 Save Cleaned News

Persist the cleaned Yahoo Finance news dataset so it can be reused by later notebook stages and Python modules.

## 6. Exploratory Data Inspection

Summarize the cleaned news dataset so you can confirm the pipeline output at a glance.

## 7. Sentiment Analysis with FinBERT

Load the FinBERT model once, score each headline, and enrich the news DataFrame with sentiment fields for downstream analysis.

In [7]:
print(f"Number of news articles: {len(news_cleaned_df)}")
print("\nMissing values:")
print(news_cleaned_df.isna().sum().to_string())
print(f"\nDuplicate headlines: {news_cleaned_df['headline'].duplicated().sum()}")
print("\nData types:")
print(news_cleaned_df.dtypes.to_string())
print(
    f"\nDate range: {news_cleaned_df['published_date'].min()} to {news_cleaned_df['published_date'].max()}"
)
print("\nCleaned dataset preview:")
print(news_cleaned_df.head().to_string(index=False))


Number of news articles: 10

Missing values:
headline          0
publisher         0
published_date    0
article_url       0

Duplicate headlines: 0

Data types:
headline                  string
publisher                 string
published_date    datetime64[us]
article_url               string

Date range: 2026-07-14 20:57:29 to 2026-07-15 10:10:00

Cleaned dataset preview:
                                                                                headline           publisher      published_date                                                                                                                      article_url
                         Chinese AI model developer DeepSeek preps for IPO: What to know Yahoo Finance Video 2026-07-14 20:57:29                    https://finance.yahoo.com/video/chinese-ai-model-developer-deepseek-preps-for-ipo-what-to-know-205729630.html
      New York enacts statewide moratorium on AI data center construction: A closer look Yahoo Finance Video

In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline

MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

sentiment_pipeline = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [12]:
# Work from the cleaned news data for sentiment analysis and keep the original frame enriched in place.
news_df = news_cleaned_df.copy()

# Apply FinBERT to every headline and expand the returned dictionaries into columns.
sentiment_results = news_df["headline"].apply(analyze_sentiment).apply(pd.Series)
news_df[["sentiment_label", "confidence", "sentiment_score"]] = sentiment_results

news_df[["headline", "sentiment_label", "confidence", "sentiment_score"]].head()

,headline,sentiment_label,confidence,sentiment_score
0,Chinese AI model developer DeepSeek preps for ...,neutral,0.941396,0.0
1,New York enacts statewide moratorium on AI dat...,neutral,0.871632,0.0
2,Can Shiba Inu Reach $1 by 2027? The Answer Wil...,neutral,0.757939,0.0
3,The Stock Market Just Flashed a Clear Warning ...,neutral,0.914913,0.0
4,The Vanguard S&P 500 ETF Is a Great Investment...,neutral,0.865758,0.0


In [13]:
def analyze_sentiment(text):
    """Analyze a single headline with FinBERT and return sentiment metrics."""
    neutral_result = {
        "label": "neutral",
        "confidence": 0.0,
        "score": 0.0,
    }

    if not isinstance(text, str):
        return neutral_result

    cleaned_text = text.strip()
    if not cleaned_text:
        return neutral_result

    try:
        result = sentiment_pipeline(cleaned_text, truncation=True)[0]
        raw_label = str(result["label"]).lower()
        confidence = float(result["score"])

        label_map = {
            "positive": "positive",
            "negative": "negative",
            "neutral": "neutral",
            "label_2": "positive",
            "label_1": "neutral",
            "label_0": "negative",
        }
        label = label_map.get(raw_label, raw_label)

        if label == "positive":
            score = confidence
        elif label == "negative":
            score = -confidence
        else:
            score = 0.0

        return {
            "label": label,
            "confidence": confidence,
            "score": score,
        }
    except Exception:
        return neutral_result

In [14]:
news_df.head()

,headline,publisher,published_date,article_url,sentiment_label,confidence,sentiment_score
0,Chinese AI model developer DeepSeek preps for ...,Yahoo Finance Video,2026-07-14 20:57:29,https://finance.yahoo.com/video/chinese-ai-mod...,neutral,0.941396,0.0
1,New York enacts statewide moratorium on AI dat...,Yahoo Finance Video,2026-07-14 21:09:12,https://finance.yahoo.com/video/new-york-enact...,neutral,0.871632,0.0
2,Can Shiba Inu Reach $1 by 2027? The Answer Wil...,Motley Fool,2026-07-15 09:41:00,https://finance.yahoo.com/markets/crypto/artic...,neutral,0.757939,0.0
3,The Stock Market Just Flashed a Clear Warning ...,Motley Fool,2026-07-15 09:50:00,https://finance.yahoo.com/markets/stocks/artic...,neutral,0.914913,0.0
4,The Vanguard S&P 500 ETF Is a Great Investment...,Motley Fool,2026-07-15 09:56:00,https://finance.yahoo.com/markets/stocks/artic...,neutral,0.865758,0.0


In [15]:
news_df["sentiment_label"].value_counts()

sentiment_label
neutral     8
positive    1
negative    1
Name: count, dtype: int64

In [16]:
news_df["sentiment_score"].describe()

count    10.000000
mean      0.029778
std       0.296729
min      -0.466320
25%       0.000000
50%       0.000000
75%       0.000000
max       0.764099
Name: sentiment_score, dtype: float64

In [17]:
# Ensure the published date column is a datetime type before extracting the calendar date.
daily_source_df = news_df.copy()
daily_source_df["published_date"] = pd.to_datetime(daily_source_df["published_date"], errors="coerce")
daily_source_df = daily_source_df.dropna(subset=["published_date", "sentiment_score", "confidence"])

# Extract the date component and aggregate article-level sentiment into daily metrics.
daily_sentiment_df = (
    daily_source_df.assign(date=daily_source_df["published_date"].dt.date)
    .groupby("date", as_index=False)
    .agg(
        avg_sentiment=("sentiment_score", "mean"),
        max_sentiment=("sentiment_score", "max"),
        min_sentiment=("sentiment_score", "min"),
        median_sentiment=("sentiment_score", "median"),
        sentiment_std=("sentiment_score", "std"),
        avg_confidence=("confidence", "mean"),
        article_count=("headline", "size"),
    )
    .sort_values("date")
    .reset_index(drop=True)
)

daily_sentiment_df

,date,avg_sentiment,max_sentiment,min_sentiment,median_sentiment,sentiment_std,avg_confidence,article_count
0,2026-07-14,0.000000,0.000000,0.00000,0.0,0.000000,0.906514,2
1,2026-07-15,0.037222,0.764099,-0.46632,0.0,0.335988,0.683699,8


In [18]:
print(daily_sentiment_df.head())
print()

daily_sentiment_df.info()
print()

print(daily_sentiment_df.describe())
print()

print(f"Total number of days: {daily_sentiment_df['date'].nunique()}")
print(f"Earliest date: {daily_sentiment_df['date'].min()}")
print(f"Latest date: {daily_sentiment_df['date'].max()}")
print()

highest_avg_day = daily_sentiment_df.loc[daily_sentiment_df['avg_sentiment'].idxmax()]
lowest_avg_day = daily_sentiment_df.loc[daily_sentiment_df['avg_sentiment'].idxmin()]
most_articles_day = daily_sentiment_df.loc[daily_sentiment_df['article_count'].idxmax()]

print(f"Day with highest average sentiment: {highest_avg_day['date']} ({highest_avg_day['avg_sentiment']:.6f})")
print(f"Day with lowest average sentiment: {lowest_avg_day['date']} ({lowest_avg_day['avg_sentiment']:.6f})")
print(f"Day with the largest number of news articles: {most_articles_day['date']} ({int(most_articles_day['article_count'])})")

         date  avg_sentiment  max_sentiment  min_sentiment  median_sentiment  sentiment_std  avg_confidence  \
0  2026-07-14       0.000000       0.000000        0.00000               0.0       0.000000        0.906514   
1  2026-07-15       0.037222       0.764099       -0.46632               0.0       0.335988        0.683699   

   article_count  
0              2  
1              8  

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date              2 non-null      object 
 1   avg_sentiment     2 non-null      float64
 2   max_sentiment     2 non-null      float64
 3   min_sentiment     2 non-null      float64
 4   median_sentiment  2 non-null      float64
 5   sentiment_std     2 non-null      float64
 6   avg_confidence    2 non-null      float64
 7   article_count     2 non-null      int64  
dtypes: float64(6), int64(1), object(1)
memory us

## 8.1 Validation

Review the daily sentiment dataset and inspect the most important summary statistics.

In [21]:
from datetime import date
import warnings

import numpy as np
import pandas as pd


if "ticker" not in globals():
    ticker = "NVDA"
if "start_date" not in globals():
    start_date = "2020-01-01"
if "end_date" not in globals():
    end_date = pd.Timestamp.today().strftime("%Y-%m-%d")


def _normalize_column_name(column: object) -> str:
    """Return a stable, lowercase snake_case column name."""
    if isinstance(column, tuple):
        flattened_parts = [str(part).strip() for part in column if str(part).strip()]
        column_name = flattened_parts[0] if flattened_parts else ""
    else:
        column_name = str(column)

    return column_name.strip().lower().replace(" ", "_")


def _coerce_date_column(frame: pd.DataFrame, column_name: str = "date") -> pd.Series:
    """Convert a date-like column to Python `datetime.date` objects."""
    coerced_dates = pd.to_datetime(frame[column_name], errors="coerce")
    return coerced_dates.dt.date


def _weighted_average(group: pd.DataFrame, value_column: str, weight_column: str = "article_count") -> float:
    """Compute a stable weighted average for merged sentiment metrics."""
    value_series = pd.to_numeric(group[value_column], errors="coerce")
    if weight_column in group.columns:
        weight_series = pd.to_numeric(group[weight_column], errors="coerce").fillna(0.0)
    else:
        weight_series = pd.Series(1.0, index=group.index)

    valid_mask = value_series.notna() & weight_series.notna()
    if not valid_mask.any():
        return float(0.0)

    valid_values = value_series.loc[valid_mask]
    valid_weights = weight_series.loc[valid_mask]
    total_weight = float(valid_weights.sum())
    if total_weight == 0.0:
        return float(valid_values.mean())

    return float(np.average(valid_values, weights=valid_weights))


def engineer_stock_features(df: pd.DataFrame) -> pd.DataFrame:
    """Engineer vectorized stock-market features from a raw or yfinance-style dataframe.

    The function flattens any MultiIndex columns returned by yfinance, standardizes
    all column names to lowercase snake_case, coerces the date column to Python
    `datetime.date` objects, and computes the core price and volume features used
    later in the modeling pipeline.
    """
    if df.empty:
        raise ValueError("stock dataframe is empty; cannot engineer features.")

    enriched_df = df.copy(deep=True)
    enriched_df.columns = [_normalize_column_name(column) for column in enriched_df.columns]

    if "date" not in enriched_df.columns:
        if isinstance(enriched_df.index, pd.DatetimeIndex):
            enriched_df = enriched_df.reset_index()
            enriched_df.columns = [_normalize_column_name(column) for column in enriched_df.columns]
        else:
            raise KeyError("A date column was not found in the stock dataframe.")

    if "date" not in enriched_df.columns:
        raise KeyError("A date column was not found in the stock dataframe after normalization.")

    enriched_df["date"] = _coerce_date_column(enriched_df, "date")
    enriched_df = enriched_df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

    required_columns = {"close", "volume"}
    missing_columns = required_columns.difference(enriched_df.columns)
    if missing_columns:
        raise KeyError(f"Missing required stock columns: {sorted(missing_columns)}")

    for numeric_column in ("open", "high", "low", "close", "volume"):
        if numeric_column in enriched_df.columns:
            enriched_df[numeric_column] = pd.to_numeric(enriched_df[numeric_column], errors="coerce")

    enriched_df["daily_return"] = enriched_df["close"].pct_change()
    enriched_df["volume_change"] = enriched_df["volume"].pct_change()
    enriched_df["sma_5"] = enriched_df["close"].rolling(window=5, min_periods=5).mean()
    enriched_df["sma_20"] = enriched_df["close"].rolling(window=20, min_periods=20).mean()
    enriched_df["rolling_volatility_10"] = enriched_df["daily_return"].rolling(window=10, min_periods=10).std()

    return enriched_df


def merge_and_align_datasets(stock_df: pd.DataFrame, sentiment_df: pd.DataFrame) -> pd.DataFrame:
    """Merge engineered stock features with sentiment while rolling closed-day sentiment forward.

    The merge is designed for market data: weekend and holiday sentiment is mapped to
    the next available trading session, while valid trading days with no news are
    explicitly filled with neutral sentiment values instead of reusing historical data.
    """
    if stock_df.empty:
        raise ValueError("stock_df is empty; cannot merge datasets.")
    if sentiment_df.empty:
        raise ValueError("sentiment_df is empty; cannot merge datasets.")

    stock_working_df = stock_df.copy(deep=True)
    sentiment_working_df = sentiment_df.copy(deep=True)

    stock_working_df.columns = [_normalize_column_name(column) for column in stock_working_df.columns]
    sentiment_working_df.columns = [_normalize_column_name(column) for column in sentiment_working_df.columns]

    if "date" not in stock_working_df.columns:
        raise KeyError("stock_df must contain a date column.")
    if "date" not in sentiment_working_df.columns:
        raise KeyError("sentiment_df must contain a date column.")

    stock_working_df["date"] = _coerce_date_column(stock_working_df, "date")
    sentiment_working_df["date"] = _coerce_date_column(sentiment_working_df, "date")

    stock_working_df = (
        stock_working_df.dropna(subset=["date"])
        .sort_values("date")
        .drop_duplicates(subset=["date"], keep="last")
        .reset_index(drop=True)
    )
    sentiment_working_df = (
        sentiment_working_df.dropna(subset=["date"])
        .sort_values("date")
        .reset_index(drop=True)
    )

    sentiment_columns = [column for column in sentiment_working_df.columns if column != "date"]
    if not sentiment_columns:
        raise ValueError("sentiment_df does not contain any sentiment features to merge.")

    stock_date_frame = pd.DataFrame({"trade_date": pd.to_datetime(stock_working_df["date"])})
    sentiment_alignment_frame = sentiment_working_df.copy(deep=True)
    sentiment_alignment_frame["date"] = pd.to_datetime(sentiment_alignment_frame["date"])

    print(f"[Phase 8] Stock input shape: {stock_df.shape}")
    print(f"[Phase 8] Sentiment input shape: {sentiment_df.shape}")
    print(f"[Phase 8] Clean stock shape before merge: {stock_working_df.shape}")
    print(f"[Phase 8] Clean sentiment shape before merge: {sentiment_working_df.shape}")

    mapped_sentiment_df = pd.merge_asof(
        sentiment_alignment_frame.sort_values("date"),
        stock_date_frame.sort_values("trade_date"),
        left_on="date",
        right_on="trade_date",
        direction="forward",
    )

    mapped_sentiment_df = mapped_sentiment_df.dropna(subset=["trade_date"]).copy()
    mapped_sentiment_df["trade_date"] = pd.to_datetime(mapped_sentiment_df["trade_date"]).dt.date
    weekend_or_holiday_roll_count = int((pd.to_datetime(mapped_sentiment_df["date"]).dt.date != mapped_sentiment_df["trade_date"]).sum())

    aligned_sentiment_rows: list[dict[str, object]] = []
    for trade_date, trade_group in mapped_sentiment_df.groupby("trade_date", sort=True):
        aligned_row: dict[str, object] = {"date": trade_date}

        if "article_count" in trade_group.columns:
            aligned_row["article_count"] = int(pd.to_numeric(trade_group["article_count"], errors="coerce").fillna(0).sum())

        for sentiment_column in sentiment_columns:
            if sentiment_column == "article_count":
                continue
            if sentiment_column in {"avg_sentiment", "avg_confidence"}:
                aligned_row[sentiment_column] = _weighted_average(trade_group, sentiment_column)
            elif sentiment_column == "max_sentiment":
                aligned_row[sentiment_column] = float(pd.to_numeric(trade_group[sentiment_column], errors="coerce").max())
            elif sentiment_column == "min_sentiment":
                aligned_row[sentiment_column] = float(pd.to_numeric(trade_group[sentiment_column], errors="coerce").min())
            elif sentiment_column == "median_sentiment":
                aligned_row[sentiment_column] = float(pd.to_numeric(trade_group[sentiment_column], errors="coerce").median())
            else:
                aligned_row[sentiment_column] = float(pd.to_numeric(trade_group[sentiment_column], errors="coerce").mean())

        aligned_sentiment_rows.append(aligned_row)

    if aligned_sentiment_rows:
        aligned_sentiment_df = pd.DataFrame(aligned_sentiment_rows)
    else:
        aligned_sentiment_df = pd.DataFrame(columns=["date", *sentiment_columns])

    outer_merged_df = pd.merge(
        stock_working_df,
        aligned_sentiment_df,
        on="date",
        how="outer",
        indicator=True,
        sort=True,
    )

    final_df = outer_merged_df.loc[outer_merged_df["close"].notna()].copy()
    final_df = final_df.sort_values("date").reset_index(drop=True)

    sentiment_fill_columns = [column for column in sentiment_columns if column in final_df.columns]
    for column in sentiment_fill_columns:
        if column == "article_count":
            final_df[column] = pd.to_numeric(final_df[column], errors="coerce").fillna(0).astype(int)
        else:
            final_df[column] = pd.to_numeric(final_df[column], errors="coerce").fillna(0.0).astype(float)

    for column in ("daily_return", "volume_change", "sma_5", "sma_20", "rolling_volatility_10"):
        if column in final_df.columns:
            final_df[column] = pd.to_numeric(final_df[column], errors="coerce")

    print(f"[Phase 8] Mapped sentiment rows: {mapped_sentiment_df.shape[0]}")
    print(f"[Phase 8] Weekend/holiday rows rolled forward: {weekend_or_holiday_roll_count}")
    print(f"[Phase 8] Outer merge shape: {outer_merged_df.shape}")
    print(f"[Phase 8] Final aligned dataset shape: {final_df.shape}")

    if final_df.shape[0] != stock_working_df.shape[0]:
        warnings.warn(
            "Final merged dataset row count differs from the stock trading calendar. "
            f"Stock rows: {stock_working_df.shape[0]}, final rows: {final_df.shape[0]}."
        )

    stock_loss_ratio = 1.0 - (final_df.shape[0] / max(stock_working_df.shape[0], 1))
    if stock_loss_ratio > 0.05:
        warnings.warn(
            f"More than 5% of trading rows were lost during alignment ({stock_loss_ratio:.2%})."
        )

    return final_df


if "stock_df" not in globals():
    stock_df = yf.download(
        tickers=ticker,
        start=start_date,
        end=end_date,
        auto_adjust=False,
        progress=False,
    )
    if stock_df.empty:
        raise ValueError(f"No stock data returned for {ticker}.")
    stock_df = stock_df.reset_index()
    stock_df = stock_df.loc[:, ["Date", "Open", "High", "Low", "Close", "Volume"]]
    print("[Phase 7] stock_df was not defined, so it was downloaded in-cell.")

if "daily_sentiment_df" not in globals():
    if "news_df" not in globals() and "news_cleaned_df" not in globals():
        raise NameError(
            "daily_sentiment_df is not defined, and no news dataframe is available to rebuild it. Run the earlier news and sentiment cells first."
        )

    news_source_df = news_df.copy(deep=True) if "news_df" in globals() else news_cleaned_df.copy(deep=True)
    if not {"sentiment_score", "confidence"}.issubset(news_source_df.columns):
        if "analyze_sentiment" not in globals():
            raise NameError(
                "daily_sentiment_df is not defined, and analyze_sentiment is unavailable to rebuild it. Run the FinBERT cells first."
            )
        sentiment_results = news_source_df["headline"].apply(analyze_sentiment).apply(pd.Series)
        news_source_df[["sentiment_label", "confidence", "sentiment_score"]] = sentiment_results

    daily_source_df = news_source_df.copy(deep=True)
    daily_source_df["published_date"] = pd.to_datetime(daily_source_df["published_date"], errors="coerce")
    daily_source_df = daily_source_df.dropna(subset=["published_date", "sentiment_score", "confidence"])
    daily_sentiment_df = (
        daily_source_df.assign(date=daily_source_df["published_date"].dt.date)
        .groupby("date", as_index=False)
        .agg(
            avg_sentiment=("sentiment_score", "mean"),
            max_sentiment=("sentiment_score", "max"),
            min_sentiment=("sentiment_score", "min"),
            median_sentiment=("sentiment_score", "median"),
            sentiment_std=("sentiment_score", "std"),
            avg_confidence=("confidence", "mean"),
            article_count=("headline", "size"),
        )
        .sort_values("date")
        .reset_index(drop=True)
    )
    print("[Phase 8] daily_sentiment_df was not defined, so it was rebuilt in-cell.")

print("Executing Phase 7: Stock Feature Engineering...")
engineered_stock_df = engineer_stock_features(stock_df)
print(f"[Phase 7] Input stock shape: {stock_df.shape}")
print(f"[Phase 7] Engineered stock shape: {engineered_stock_df.shape}")
print(engineered_stock_df.head().to_string(index=False))
print()

print("Executing Phase 8: Dataset Merging & Alignment Strategy...")
final_dataset_df = merge_and_align_datasets(engineered_stock_df, daily_sentiment_df)
print(final_dataset_df.head().to_string(index=False))
print()
print("Final dataset info:")
final_dataset_df.info()
print()
print(f"Final dataset shape: {final_dataset_df.shape}")
print(f"Trading dates covered: {final_dataset_df['date'].nunique()}")

Executing Phase 7: Stock Feature Engineering...
[Phase 7] Input stock shape: (1640, 6)
[Phase 7] Engineered stock shape: (1640, 11)
      date    open    high     low   close    volume  daily_return  volume_change   sma_5  sma_20  rolling_volatility_10
2020-01-02 5.96875 5.99775 5.91800 5.99775 237536000           NaN            NaN     NaN     NaN                    NaN
2020-01-03 5.87750 5.94575 5.85250 5.90175 205384000     -0.016006      -0.135356     NaN     NaN                    NaN
2020-01-06 5.80800 5.93175 5.78175 5.92650 262636000      0.004194       0.278756     NaN     NaN                    NaN
2020-01-07 5.95500 6.04425 5.90975 5.99825 314856000      0.012107       0.198830     NaN     NaN                    NaN
2020-01-08 5.99400 6.05100 5.95375 6.00950 277108000      0.001876      -0.119890 5.96675     NaN                    NaN

Executing Phase 8: Dataset Merging & Alignment Strategy...
[Phase 8] Stock input shape: (1640, 11)
[Phase 8] Sentiment input shape: (2, 8)
[P

/var/folders/3y/_pxxfsdn4pn_nfhn8p6_wzs80000gn/T/ipykernel_45620/123232770.py:219: UserWarning: Final merged dataset row count differs from the stock trading calendar. Stock rows: 1640, final rows: 1639.
  warnings.warn(


In [31]:
# (Already in your notebook)
engineered_stock_df = engineer_stock_features(stock_df)

In [32]:
# ============================================================
# Synthetic News Generator for Sentiment-Price Correlation Tool
# ============================================================
# Input:
#   engineered_stock_df
#       Required columns:
#       ['date', 'close', 'daily_return', 'volume']
#
# Output:
#   synthetic_news_df
#       Columns:
#       ['headline', 'publisher', 'published_date', 'article_url']
#
# Generates exactly 1,000 synthetic news articles whose sentiment
# is correlated with the underlying daily stock return.
# ============================================================

import pandas as pd
import numpy as np
import random

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
NUM_NEWS = 1000
STOCK_NAME = "NVIDIA"
STOCK_TICKER = "NVDA"

random.seed(42)
np.random.seed(42)

# ------------------------------------------------------------
# Publishers
# ------------------------------------------------------------
publishers = {
    "Reuters": "https://www.reuters.com/business/",
    "Bloomberg": "https://www.bloomberg.com/news/articles/",
    "Wall Street Journal": "https://www.wsj.com/articles/",
    "CNBC": "https://www.cnbc.com/",
    "MarketWatch": "https://www.marketwatch.com/story/",
    "Financial Times": "https://www.ft.com/content/",
    "Yahoo Finance": "https://finance.yahoo.com/news/",
    "The Motley Fool": "https://www.fool.com/investing/",
    "Barron's": "https://www.barrons.com/articles/",
}

# ------------------------------------------------------------
# Dynamic Vocabulary
# ------------------------------------------------------------

bull_subjects = [
    "earnings",
    "AI demand",
    "data center business",
    "GPU shipments",
    "cloud partnerships",
    "enterprise adoption",
    "chip sales",
    "new product lineup",
    "quarterly revenue",
    "market expansion",
    "institutional buying",
    "profit margins",
    "guidance",
    "analyst upgrades",
]

bull_actions = [
    "surges after",
    "jumps following",
    "rallies as",
    "climbs on",
    "extends gains after",
    "advances following",
    "moves higher after",
]

bull_events = [
    "record quarterly earnings",
    "better-than-expected guidance",
    "strong AI demand",
    "major cloud partnership",
    "new GPU launch",
    "record revenue growth",
    "positive analyst revisions",
    "large enterprise contracts",
    "expanding global demand",
    "accelerating data-center sales",
    "higher operating margins",
]

bear_subjects = [
    "supply chain",
    "regulatory concerns",
    "chip demand",
    "profit outlook",
    "AI competition",
    "production costs",
    "market sentiment",
    "guidance",
    "investor confidence",
    "valuation",
    "pricing pressure",
]

bear_actions = [
    "falls after",
    "slides as",
    "drops following",
    "declines amid",
    "retreats after",
    "weakens as",
]

bear_events = [
    "disappointing guidance",
    "regulatory scrutiny",
    "supply chain bottlenecks",
    "slower enterprise demand",
    "profit-taking by investors",
    "competitor pressure",
    "rising production costs",
    "weaker-than-expected sales",
    "macroeconomic uncertainty",
    "inventory concerns",
]

neutral_events = [
    "trades sideways ahead of economic data",
    "holds steady before earnings",
    "remains range-bound during quiet trading",
    "sees limited movement as investors wait",
    "consolidates after recent volatility",
    "shows muted trading activity",
    "moves little amid mixed market signals",
    "awaits Federal Reserve commentary",
    "remains stable despite broader market moves",
    "trades in a narrow range",
]

# ------------------------------------------------------------
# Headline Generator
# ------------------------------------------------------------

def generate_headline(daily_return):
    """
    Create sentiment-aligned headline based on daily return.
    """

    if daily_return > 0.015:
        return (
            f"{STOCK_NAME} "
            f"{random.choice(bull_actions)} "
            f"{random.choice(bull_events)} "
            f"as {random.choice(bull_subjects)} strengthens"
        )

    elif daily_return < -0.015:
        return (
            f"{STOCK_NAME} "
            f"{random.choice(bear_actions)} "
            f"{random.choice(bear_events)} "
            f"amid {random.choice(bear_subjects)} concerns"
        )

    else:
        return (
            f"{STOCK_NAME} "
            f"{random.choice(neutral_events)}"
        )

# ------------------------------------------------------------
# URL Generator
# ------------------------------------------------------------

def generate_url(base_url):
    random_id = ''.join(
        random.choices(
            'abcdefghijklmnopqrstuvwxyz0123456789',
            k=12
        )
    )
    return base_url + random_id

# ------------------------------------------------------------
# Prepare Stock Data
# ------------------------------------------------------------

stock_df = engineered_stock_df.copy()

stock_df["date"] = pd.to_datetime(stock_df["date"])

# Sample 1000 random trading days (with replacement)
sampled = stock_df.sample(
    n=NUM_NEWS,
    replace=True,
    random_state=42
).reset_index(drop=True)

# ------------------------------------------------------------
# Generate Synthetic News
# ------------------------------------------------------------

news_rows = []

for _, row in sampled.iterrows():

    publisher = random.choice(list(publishers.keys()))

    headline = generate_headline(row["daily_return"])

    # Random publication time during market hours
    hour = random.randint(8, 18)
    minute = random.randint(0, 59)
    second = random.randint(0, 59)

    published_datetime = (
        row["date"]
        + pd.Timedelta(
            hours=hour,
            minutes=minute,
            seconds=second
        )
    )

    news_rows.append({
        "headline": headline,
        "publisher": publisher,
        "published_date": published_datetime.strftime("%Y-%m-%d %H:%M:%S"),
        "article_url": generate_url(publishers[publisher])
    })

# ------------------------------------------------------------
# Final DataFrame
# ------------------------------------------------------------

synthetic_news_df = pd.DataFrame(
    news_rows,
    columns=[
        "headline",
        "publisher",
        "published_date",
        "article_url",
    ]
)

# ------------------------------------------------------------
# Verify Output
# ------------------------------------------------------------

print(f"Generated {len(synthetic_news_df)} synthetic news articles.\n")

print(synthetic_news_df.head())

print("\nSchema:")
print(synthetic_news_df.dtypes)

Generated 1000 synthetic news articles.

                                            headline        publisher       published_date  \
0  NVIDIA surges after new GPU launch as GPU ship...        Bloomberg  2024-06-25 11:08:47   
1                    NVIDIA trades in a narrow range  The Motley Fool  2025-10-22 12:51:55   
2  NVIDIA remains stable despite broader market m...  The Motley Fool  2023-06-02 09:59:24   
3  NVIDIA rallies as large enterprise contracts a...    Yahoo Finance  2025-02-26 13:10:23   
4                NVIDIA shows muted trading activity             CNBC  2024-07-01 08:14:52   

                                         article_url  
0  https://www.bloomberg.com/news/articles/d0tvbd...  
1        https://www.fool.com/investing/a3zmf8mdd4v3  
2        https://www.fool.com/investing/ckw5ngcx1945  
3        https://finance.yahoo.com/news/myzycwtiqj7y  
4                  https://www.cnbc.com/bljh75lxo6qj  

Schema:
headline          str
publisher         str
published_d

In [33]:
# Create a copy to track progress
scored_synthetic_news_df = synthetic_news_df.copy()

print("Running 1,000 headlines through FinBERT... (This may take 1-2 minutes)")
# Apply FinBERT to the synthetic headlines
sentiment_results = scored_synthetic_news_df["headline"].apply(analyze_sentiment).apply(pd.Series)
scored_synthetic_news_df[["sentiment_label", "confidence", "sentiment_score"]] = sentiment_results

print("FinBERT inference complete!")
scored_synthetic_news_df.head()

Running 1,000 headlines through FinBERT... (This may take 1-2 minutes)
FinBERT inference complete!


,headline,publisher,published_date,article_url,sentiment_label,confidence,sentiment_score
0,NVIDIA surges after new GPU launch as GPU ship...,Bloomberg,2024-06-25 11:08:47,https://www.bloomberg.com/news/articles/d0tvbd...,positive,0.940200,0.940200
1,NVIDIA trades in a narrow range,The Motley Fool,2025-10-22 12:51:55,https://www.fool.com/investing/a3zmf8mdd4v3,neutral,0.799517,0.000000
2,NVIDIA remains stable despite broader market m...,The Motley Fool,2023-06-02 09:59:24,https://www.fool.com/investing/ckw5ngcx1945,positive,0.939184,0.939184
3,NVIDIA rallies as large enterprise contracts a...,Yahoo Finance,2025-02-26 13:10:23,https://finance.yahoo.com/news/myzycwtiqj7y,positive,0.898593,0.898593
4,NVIDIA shows muted trading activity,CNBC,2024-07-01 08:14:52,https://www.cnbc.com/bljh75lxo6qj,negative,0.925193,-0.925193


In [34]:
# Aggregate the scored synthetic news into daily statistics
daily_source_df = scored_synthetic_news_df.copy()
daily_source_df["published_date"] = pd.to_datetime(daily_source_df["published_date"])

daily_sentiment_df = (
    daily_source_df.assign(date=daily_source_df["published_date"].dt.date)
    .groupby("date", as_index=False)
    .agg(
        avg_sentiment=("sentiment_score", "mean"),
        max_sentiment=("sentiment_score", "max"),
        min_sentiment=("sentiment_score", "min"),
        median_sentiment=("sentiment_score", "median"),
        sentiment_std=("sentiment_score", "std"),
        avg_confidence=("confidence", "mean"),
        article_count=("headline", "size"),
    )
    .sort_values("date")
    .reset_index(drop=True)
)

## 8. Daily Sentiment Aggregation

Aggregate article-level sentiment into a daily sentiment dataset without merging stock data yet.

In [35]:
from typing import Dict, Any
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr

def perform_statistical_analysis(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Computes Pearson, Spearman, Lag, and Rolling correlations between 
    stock returns and news sentiment.
    
    Time Complexity: 
        - Pearson/Spearman: O(N log N) due to rank sorting in Spearman.
        - Lag Correlation: O(L * N) where L is the number of lags.
        - Rolling Correlation: O(W * N) where W is the window size.
    """
    analysis_results = {}
    
    # Create a local copy and drop any remaining NaNs in our target analysis columns
    analysis_df = df.copy().dropna(subset=['daily_return', 'avg_sentiment'])
    
    # 1. Global Baseline Correlations
    # We use scipy to get p-values alongside the correlation coefficients
    if len(analysis_df) > 1:
        # Check if we have variance in both series to prevent division by zero in correlation math
        if analysis_df['avg_sentiment'].nunique() > 1 and analysis_df['daily_return'].nunique() > 1:
            p_coef, p_val = pearsonr(analysis_df['avg_sentiment'], analysis_df['daily_return'])
            s_coef, s_val = spearmanr(analysis_df['avg_sentiment'], analysis_df['daily_return'])
        else:
            print("⚠️ Warning: Constant data detected in sentiment or returns. Correlation cannot be calculated.")
            p_coef, p_val, s_coef, s_val = np.nan, np.nan, np.nan, np.nan
    else:
        p_coef, p_val, s_coef, s_val = np.nan, np.nan, np.nan, np.nan
        
    analysis_results['pearson'] = {'coefficient': p_coef, 'p_value': p_val}
    analysis_results['spearman'] = {'coefficient': s_coef, 'p_value': s_val}
    
    # 2. Lag Correlation Analysis (Predictive Power)
    # Lag -1: Yesterday's sentiment vs Today's returns (Does sentiment predict returns?)
    # Lag +1: Today's sentiment vs Yesterday's returns (Does price drive sentiment?)
    lags = [-3, -2, -1, 0, 1, 2, 3]
    lag_correlations = {}
    
    for lag in lags:
        # Shift the sentiment series
        shifted_sentiment = analysis_df['avg_sentiment'].shift(lag)
        # Calculate correlation with daily returns
        corr = analysis_df['daily_return'].corr(shifted_sentiment)
        lag_correlations[f"lag_{lag}"] = corr
        
    analysis_results['lag_correlations'] = lag_correlations
    
    # 3. Rolling Correlation (Market Regimes)
    # We compute a rolling 30-day Pearson correlation
    rolling_window = 30
    if len(analysis_df) >= rolling_window:
        analysis_df['rolling_corr_30d'] = (
            analysis_df['daily_return']
            .rolling(window=rolling_window)
            .corr(analysis_df['avg_sentiment'])
        )
        # Store the rolling series back to the original DataFrame to help our visualization phase later
        df['rolling_sentiment_price_corr_30d'] = analysis_df['rolling_corr_30d']
    else:
        df['rolling_sentiment_price_corr_30d'] = np.nan
    
    # 4. Print Executive Summary
    print("\n" + "="*50)
    print("       FINANCIAL SENTIMENT STATISTICAL SUMMARY")
    print("="*50)
    print(f"Total trading days with sentiment analyzed: {len(analysis_df)}")
    print(f"Overall Pearson Correlation:  {p_coef:.6f} (p-value: {p_val:.4e})" if not np.isnan(p_coef) else "Overall Pearson Correlation:  N/A")
    print(f"Overall Spearman Correlation: {s_coef:.6f} (p-value: {s_val:.4e})" if not np.isnan(s_coef) else "Overall Spearman Correlation: N/A")
    print("\n--- Lag Correlation Map (Predictive Analysis) ---")
    print("Negative lag = Sentiment leads Price (Predictive)")
    print("Positive lag = Sentiment lags Price  (Reactive)")
    for lag, corr in lag_correlations.items():
        val_str = f"{corr:+.6f}" if not np.isnan(corr) else "N/A"
        print(f"  {lag.replace('_', ' '):<8}: {val_str}")
    print("="*50 + "\n")
    
    return analysis_results


# --- 🚀 SMART EXECUTION WRAPPER ---
# Dynamically search your active Jupyter workspace for the Phase 8 merged DataFrame
merged_df = None
for var_name, var_val in list(globals().items()):
    if isinstance(var_val, pd.DataFrame):
        # Identify the DataFrame by checking for our custom engineered columns
        if 'daily_return' in var_val.columns and 'avg_sentiment' in var_val.columns:
            merged_df = var_val
            print(f"✅ Auto-detected your merged Phase 8 DataFrame in variable: '{var_name}'")
            break

if merged_df is not None:
    statistical_results = perform_statistical_analysis(merged_df)
else:
    print("❌ Error: Could not locate your merged Phase 8 DataFrame in the workspace.")
    print("Please ensure you have run the Phase 8 cell before executing this analysis.")

✅ Auto-detected your merged Phase 8 DataFrame in variable: 'final_dataset_df'
⚠️ Warning: Constant data detected in sentiment or returns. Correlation cannot be calculated.

       FINANCIAL SENTIMENT STATISTICAL SUMMARY
Total trading days with sentiment analyzed: 1638
Overall Pearson Correlation:  N/A
Overall Spearman Correlation: N/A

--- Lag Correlation Map (Predictive Analysis) ---
Negative lag = Sentiment leads Price (Predictive)
Positive lag = Sentiment lags Price  (Reactive)
  lag -3  : N/A
  lag -2  : N/A
  lag -1  : N/A
  lag 0   : N/A
  lag 1   : N/A
  lag 2   : N/A
  lag 3   : N/A



/Users/shreyasvij/Projects/Sentiment-Price-Correlation/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/shreyasvij/Projects/Sentiment-Price-Correlation/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [36]:
import warnings

import pandas as pd


def merge_and_align_datasets(stock_df: pd.DataFrame, sentiment_df: pd.DataFrame) -> pd.DataFrame:
    """Merge stock and sentiment datasets on standardized ISO date strings.

    The function preserves trading days, rolls sentiment from weekend and holiday
    dates forward to the next available market session, and fills genuine no-news
    trading days with neutral defaults.
    """
    if stock_df.empty:
        raise ValueError("stock_df is empty; cannot merge datasets.")
    if sentiment_df.empty:
        raise ValueError("sentiment_df is empty; cannot merge datasets.")

    stock_working_df = stock_df.copy(deep=True)
    sentiment_working_df = sentiment_df.copy(deep=True)

    if "date" not in stock_working_df.columns:
        raise KeyError("stock_df must contain a 'date' column.")
    if "date" not in sentiment_working_df.columns:
        raise KeyError("sentiment_df must contain a 'date' column.")

    stock_working_df["date_str"] = pd.to_datetime(stock_working_df["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    sentiment_working_df["date_str"] = pd.to_datetime(sentiment_working_df["date"], errors="coerce").dt.strftime("%Y-%m-%d")

    stock_working_df = (
        stock_working_df.dropna(subset=["date_str"])
        .drop_duplicates(subset=["date_str"], keep="last")
        .drop(columns=["date"])
    )
    sentiment_working_df = (
        sentiment_working_df.dropna(subset=["date_str"])
        .drop_duplicates(subset=["date_str"], keep="last")
        .drop(columns=["date"])
    )

    merged_df = pd.merge(
        stock_working_df,
        sentiment_working_df,
        on="date_str",
        how="outer",
        indicator=True,
        sort=True,
    ).sort_values("date_str").reset_index(drop=True)

    sentiment_columns = [
        column
        for column in [
            "avg_sentiment",
            "max_sentiment",
            "min_sentiment",
            "median_sentiment",
            "sentiment_std",
            "avg_confidence",
            "article_count",
        ]
        if column in merged_df.columns
    ]

    if sentiment_columns:
        reversed_df = merged_df.iloc[::-1].copy()
        reversed_df[sentiment_columns] = reversed_df[sentiment_columns].bfill()
        merged_df.loc[:, sentiment_columns] = reversed_df.iloc[::-1][sentiment_columns].to_numpy()

    merged_df = merged_df.loc[merged_df["_merge"] != "right_only"].copy()
    merged_df = merged_df.sort_values("date_str").reset_index(drop=True)
    merged_df["date"] = pd.to_datetime(merged_df["date_str"], errors="coerce").dt.date

    for column in [
        "avg_sentiment",
        "max_sentiment",
        "min_sentiment",
        "median_sentiment",
        "sentiment_std",
        "avg_confidence",
    ]:
        if column in merged_df.columns:
            merged_df[column] = pd.to_numeric(merged_df[column], errors="coerce").fillna(0.0).astype(float)

    if "article_count" in merged_df.columns:
        merged_df["article_count"] = pd.to_numeric(merged_df["article_count"], errors="coerce").fillna(0).astype(int)

    return merged_df.drop(columns=["date_str", "_merge"], errors="ignore")


if "engineered_stock_df" not in globals():
    raise NameError("engineered_stock_df is not available. Run the stock engineering cell first.")
if "daily_sentiment_df" not in globals():
    raise NameError("daily_sentiment_df is not available. Run the sentiment aggregation cell first.")

processed_stock_df = engineered_stock_df
final_dataset_df = merge_and_align_datasets(processed_stock_df, daily_sentiment_df)

non_zero_sentiment_count = int((final_dataset_df["avg_sentiment"] != 0).sum())
print(f"Non-zero avg_sentiment rows: {non_zero_sentiment_count}")

if non_zero_sentiment_count > 0:
    print(
        final_dataset_df.loc[
            final_dataset_df["avg_sentiment"] != 0,
            ["date", "close", "daily_return", "avg_sentiment", "article_count"],
        ].to_string(index=False)
    )
else:
    warnings.warn(
        "Validation warning: no non-zero avg_sentiment rows survived the Phase 8 merge.",
        RuntimeWarning,
    )

Non-zero avg_sentiment rows: 1404
      date      close  daily_return  avg_sentiment  article_count
2020-01-02   5.997750           NaN      -0.613020              1
2020-01-03   5.901750     -0.016006      -0.969311              2
2020-01-06   5.926500      0.004194      -0.969311              2
2020-01-07   5.998250      0.012107      -0.969311              2
2020-01-13   6.299500      0.031352       0.939879              1
2020-01-14   6.182000     -0.018652      -0.968002              1
2020-01-17   6.232000      0.001406      -0.925193              1
2020-01-21   6.198500     -0.005375      -0.925193              1
2020-01-23   6.321500      0.010995      -0.901788              1
2020-01-24   6.262000     -0.009412      -0.901788              1
2020-01-27   6.005000     -0.041041      -0.965615              1
2020-01-28   6.199250      0.032348       0.924701              1
2020-01-29   6.138500     -0.009800       0.924701              1
2020-01-31   5.910750     -0.038160      -

In [37]:
# =====================================================================
# PHASE 9.1: MULTI-DAY PIPELINE & STATISTICAL INTEGRATION TEST
# =====================================================================
# Injects mock headlines on fully settled historical business days 
# to trigger actual statistical calculations.

print("Injecting historical mock news (July 10 & July 13)...")

# 1. Mock news database (utilizing fully finalized trading dates)
mock_news_database = pd.DataFrame([
    {
        "headline": "NVIDIA chip preorders shatter expectations as AI cloud providers accelerate spending.",
        "publisher": "Wall Street Journal",
        "published_date": pd.to_datetime("2026-07-10 10:15:00"),
        "article_url": "https://finance.yahoo.com/mock-pos"
    },
    {
        "headline": "NVIDIA faces unexpected regulatory hurdles and supply chain bottleneck issues.",
        "publisher": "Reuters",
        "published_date": pd.to_datetime("2026-07-13 11:30:00"),
        "article_url": "https://finance.yahoo.com/mock-neg"
    }
])

# 2. Combine, apply FinBERT, and group
validation_news_df = pd.concat([news_cleaned_df, mock_news_database], ignore_index=True)
validation_news_df[["sentiment_label", "confidence", "sentiment_score"]] = (
    validation_news_df["headline"].apply(analyze_sentiment).apply(pd.Series)
)

validation_daily_sentiment = (
    validation_news_df.assign(date=validation_news_df["published_date"].dt.date)
    .groupby("date", as_index=False)
    .agg(
        avg_sentiment=("sentiment_score", "mean"),
        max_sentiment=("sentiment_score", "max"),
        min_sentiment=("sentiment_score", "min"),
        median_sentiment=("sentiment_score", "median"),
        sentiment_std=("sentiment_score", "std"),
        avg_confidence=("confidence", "mean"),
        article_count=("headline", "size"),
    )
    .sort_values("date")
    .reset_index(drop=True)
)

# 3. Merge with engineered stock data
test_final_dataset_df = merge_and_align_datasets(engineered_stock_df, validation_daily_sentiment)

# 4. Verify that the historical days successfully aligned
active_sentiment_days = test_final_dataset_df[test_final_dataset_df["avg_sentiment"] != 0.0]
print(f"\n[Validation] Aligned sentiment days remaining: {len(active_sentiment_days)}")

if len(active_sentiment_days) >= 2:
    print("\n🎉 SUCCESS! Aligned trading dates that survived the merge:")
    print(active_sentiment_days[['date', 'close', 'daily_return', 'avg_sentiment', 'article_count']].to_string(index=False))
    
    # 5. Run the Phase 9 statistical suite on the active dataset
    print("\nRunning statistical test suite...")
    test_stats = perform_statistical_analysis(test_final_dataset_df)
else:
    print("❌ Error: Some test rows were dropped. Check dates.")

Injecting historical mock news (July 10 & July 13)...

[Validation] Aligned sentiment days remaining: 2

🎉 SUCCESS! Aligned trading dates that survived the merge:
      date      close  daily_return  avg_sentiment  article_count
2026-07-10 210.960007      0.040339      -0.951654              1
2026-07-13 203.529999     -0.035220      -0.946648              1

Running statistical test suite...

       FINANCIAL SENTIMENT STATISTICAL SUMMARY
Total trading days with sentiment analyzed: 1638
Overall Pearson Correlation:  0.000032 (p-value: 9.9896e-01)
Overall Spearman Correlation: -0.000214 (p-value: 9.9308e-01)

--- Lag Correlation Map (Predictive Analysis) ---
Negative lag = Sentiment leads Price (Predictive)
Positive lag = Sentiment lags Price  (Reactive)
  lag -3  : -0.020301
  lag -2  : -0.013150
  lag -1  : -0.015033
  lag 0   : +0.000032
  lag 1   : +0.028545
  lag 2   : N/A
  lag 3   : N/A



/Users/shreyasvij/Projects/Sentiment-Price-Correlation/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/shreyasvij/Projects/Sentiment-Price-Correlation/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [39]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def create_interactive_dashboard(merged_df: pd.DataFrame, news_df: pd.DataFrame) -> go.Figure:
    """
    Generates a production-grade, highly interactive, dark-themed financial dashboard.
    Maps historical stock price (line) against daily sentiment (bars), using a 
    secondary Y-axis to handle scale discrepancies.
    
    Includes a smart annotation engine that filters and labels only the top 5 
    most statistically significant price-sentiment outlier events to avoid clutter.
    """
    plot_df = merged_df.copy()
    raw_news = news_df.copy()
    
    # 1. Standardize dates to strings for robust mapping
    plot_df['date_str'] = pd.to_datetime(plot_df['date']).dt.strftime('%Y-%m-%d')
    raw_news['date_str'] = pd.to_datetime(raw_news['published_date']).dt.strftime('%Y-%m-%d')
    
    # 2. Compile headlines into clean HTML lists for Plotly hover tooltips
    def compile_headlines(group):
        headlines = group['headline'].tolist()
        publishers = group['publisher'].tolist()
        formatted = []
        # Display up to 3 headlines per hover card to keep tooltips concise
        for i, (h, p) in enumerate(zip(headlines[:3], publishers[:3])):
            trunc_h = h[:60] + "..." if len(h) > 60 else h
            formatted.append(f"• <b>[{p}]</b> {trunc_h}")
        
        suffix = f"<br><i>... and {len(headlines)-3} more articles</i>" if len(headlines) > 3 else ""
        return "<br>".join(formatted) + suffix

    tooltip_lookup = raw_news.groupby('date_str').apply(compile_headlines).to_dict()
    plot_df['headline_summary'] = plot_df['date_str'].map(tooltip_lookup).fillna("No news collected for this session.")
    
    # 3. Initialize Subplots with dual Y-Axes
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # Trace 1: Stock Close Price (Primary Y-Axis)
    fig.add_trace(
        go.Scatter(
            x=plot_df['date'],
            y=plot_df['close'],
            name="NVDA Stock Price ($)",
            line=dict(color="#10B981", width=2.5),  # Emerald Green
            hovertemplate="<b>Date:</b> %{x}<br><b>Price:</b> $%{y:.2f}<extra></extra>"
        ),
        secondary_y=False
    )
    
    # Trace 2: Daily Sentiment Average (Secondary Y-Axis)
    # Conditional coloring: Soft Green for positive (>0.05), Soft Red for negative (<-0.05), Grey for neutral
    colors = np.where(plot_df['avg_sentiment'] > 0.05, '#34D399', 
             np.where(plot_df['avg_sentiment'] < -0.05, '#F87171', '#4B5563'))
    
    fig.add_trace(
        go.Bar(
            x=plot_df['date'],
            y=plot_df['avg_sentiment'],
            name="Avg Sentiment Score",
            marker_color=colors,
            opacity=0.6,
            customdata=plot_df['headline_summary'],
            hovertemplate="<b>Date:</b> %{x}<br><b>Sentiment:</b> %{y:+.4f}<br><br><b>Top News:</b><br>%{customdata}<extra></extra>"
        ),
        secondary_y=True
    )
    
    # 4. Apply Bloomberg-Slate Dark Theme Layout Styling
    fig.update_layout(
        template="plotly_dark",
        title={
            'text': "<b>NVIDIA (NVDA) Price & News Sentiment Correlation</b>",
            'y': 0.95,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': dict(size=22, color="#F3F4F6")
        },
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        margin=dict(l=60, r=60, t=100, b=50),
        plot_bgcolor="#111827",  # Deep slate gray
        paper_bgcolor="#111827",
        xaxis=dict(
            title="Date Timeline",
            gridcolor="#374151",
            rangeslider=dict(visible=True),  # Dynamic interactive timeframe slider
            type="date"
        ),
        yaxis=dict(
            title="Stock Price (USD)",
            gridcolor="#374151",
            tickprefix="$"
        ),
        yaxis2=dict(
            title="Sentiment Score (FinBERT Bounds)",
            gridcolor="rgba(0,0,0,0)",  # Prevent grid lines overlapping
            range=[-1.05, 1.05],
            tickformat="+.1f"
        )
    )
    
    # 5. Smart Annotation Engine: Isolate Top 5 Outliers
    # Identify days where absolute returns and absolute sentiment are both high
    plot_df['impact_score'] = plot_df['daily_return'].abs() * plot_df['avg_sentiment'].abs()
    top_events = plot_df.nlargest(5, 'impact_score')
    
    for _, row in top_events.iterrows():
        is_positive = row['avg_sentiment'] > 0
        label_text = "🟢 Bullish Catalyst" if is_positive else "🔴 Bearish Catalyst"
        accent_color = "#34D399" if is_positive else "#F87171"
        
        fig.add_annotation(
            x=row['date'],
            y=row['close'],
            text=label_text,
            showarrow=True,
            arrowhead=2,
            arrowcolor="#9CA3AF",
            ax=0,
            ay=-40 if is_positive else 40,  # Offset positive labels up, negative down
            bordercolor=accent_color,
            borderwidth=1,
            borderpad=4,
            bgcolor="#1F2937",
            opacity=0.95
        )
        
    return fig

# Run the dashboard utilizing your rich synthetic news dataset
# We use 'synthetic_news_df' since it contains your full 1,000 headlines dataset!
fig = create_interactive_dashboard(final_dataset_df, synthetic_news_df)
fig.show()